In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats
import statsmodels.api as sm

# Step 1: Load and preprocess the data
df = pd.read_excel("/mnt/data/CrimeDataDelete.xlsx")
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')
df = df.dropna(subset=['DATE OCC'])
monthly_counts = df['DATE OCC'].dt.to_period('M').value_counts().sort_index()
monthly_counts.index = monthly_counts.index.to_timestamp()

# Step 2: Fit an ARIMA model (you can tune order if needed)
model = ARIMA(monthly_counts, order=(1, 1, 1))
results = model.fit()

# Step 3: Extract fitted values and residuals
fitted = results.fittedvalues
residuals = results.resid

# Step 4: Plot residuals
plt.figure(figsize=(12, 5))
plt.plot(residuals)
plt.title("Residuals from ARIMA(1,1,1)")
plt.grid(True)
plt.show()

# Step 5: Residual mean check
res_mean = np.mean(residuals)
print(f"Mean of residuals: {res_mean:.4f}")

# Step 6: Autocorrelation of residuals
lags = min(20, len(residuals) // 2)  # Avoid shape mismatch
plot_acf(residuals, lags=lags)
plt.title("ACF of Residuals")
plt.show()

# Step 7: Ljung-Box test (for no autocorrelation)
lb_test = acorr_ljungbox(residuals, lags=[lags], return_df=True)
print("\nLjung-Box Test:")
print(lb_test)

# Step 8: Normality test (Histogram & Q-Q Plot)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(residuals, kde=True)
plt.title("Histogram of Residuals")

plt.subplot(1, 2, 2)
sm.qqplot(residuals, line='s')
plt.title("Q-Q Plot of Residuals")
plt.tight_layout()
plt.show()

# Shapiro-Wilk test for normality
shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f"\nShapiro-Wilk Test: W={shapiro_stat:.4f}, p={shapiro_p:.4f}")
if shapiro_p > 0.05:
    print("Residuals look approximately normal (p > 0.05)")
else:
    print("Residuals deviate from normality (p ≤ 0.05)")

# Step 9: Constant variance (Homoscedasticity)
plt.figure(figsize=(10, 4))
plt.scatter(fitted, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuals vs Fitted Values")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.grid(True)
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/CrimeDataDelete.xlsx'